# Factor Search

**Table of contents**

- Overview
- Setup
  - Authentication Token
- Query
  - Output Description
- Related Links

## Overview

This notebook demonstrates how to use the Search API to find and browse multiple emission factors that match specific criteria. It allows users to:

- Search for emission factors using keywords (e.g., "gas", "electricity")
- Filter results by location and date
- Navigate through paginated results to explore all available options
- Compare different emission factors before selecting one for calculations

The Search API is valuable for exploration and discovery, helping users understand what emission factors are available before making calculations or when they need to compare multiple methodologies. This is particularly useful when exploring available emissions factors for a specific activity type or region. The results include detailed information about each factor, including its source, scope, unit of measurement, and emissions values, presented in a paginated format for easy navigation through large result sets.

## Setup


Ensure that you have Python installed in your system. Python 3+ is required.

<b>Note:</b> To run this notebook, you must first add your credentials to `config.read('../../../auth/secrets.ini')` in the following format:


```
[EAPI]
api.pat_token = <Your Emissions API PAT token>
api.client_id = <Your GHG APIs client Id>
```

In [ ]:
# Install the packages below using pip/pip3 based on your python version.
%pip install pandas configparser IPython requests

In [1]:
import pandas as pd
import configparser
import requests
import json
from IPython.display import display as display_summary

### Authentication Token



Run the following code snippet to generate the Auth Bearer Token by using your api_key configured in secrets.ini.

In [2]:
config = configparser.RawConfigParser()
config.read(['../../../auth/secrets.ini','../../../auth/config.ini'])

EAPI_PAT_TOKEN      = config.get('EAPI', 'api.pat_token')
EAPI_TENANT_ID      = config.get('EAPI', 'api.tenant_id')

EAPI_AUTH_ENDPOINT  = config.get('EAPI', 'api.auth_endpoint')
EAPI_BASE_URL       = config.get('EAPI', 'api.base_url')
EAPI_ENDPOINT       = f"{EAPI_BASE_URL}/factor/search"

EAPI_AUTH_CLIENT_ID = 'saascore-' + EAPI_TENANT_ID
EAPI_CLIENT_ID      = 'ghgemissions-' + EAPI_TENANT_ID

auth_request_headers: dict = {}
auth_request_headers["X-IBM-Client-Id"] = EAPI_AUTH_CLIENT_ID
auth_request_headers["X-IBM-Envizi-Pat"] = EAPI_PAT_TOKEN

verify = True

auth_url = f"{EAPI_AUTH_ENDPOINT}"
              
response = requests.post(url = auth_url,
                        headers = auth_request_headers,
                        verify  = verify
                       )
if response.status_code == 200:
    jwt_token = response.text
    print("Authentication Success")
else:     
    print("Authentication Failed")
    print(response.text)

Authentication Success


## Query

The example request payload demonstrates how to search for emission factors using IBM Envizi - Emissions API by specifying location (India), a keyword ("gas"), pagination parameters (page 1, 30 results), and a reference date (January 4, 2024).

In [3]:
payload = {
  "location": {
    "country": "ind"
  },
  "activity": {
    "search": "gas"
  },
  "pagination": {
    "page": 1,
    "size": 30
  },
  "time": {
    "date": "2024-01-04"
  }
}

In [4]:
# Create the query headers
request_headers: dict = {}
request_headers["Content-Type"] = "application/json"
request_headers["x-ibm-client-id"] = EAPI_CLIENT_ID
request_headers["Authorization"] = "Bearer " + jwt_token

# Submit the request
response = requests.post(EAPI_ENDPOINT, 
                         headers = request_headers, 
                         data = json.dumps(payload))

For more information about allowable parameters for the payload, please see [Emissions API Developer Guide]().

In [5]:
if response.status_code == 200 and response.text:
    # Parse the JSON response
    response_json = response.json()
    
    # Extract the factors list
    factors_list = response_json.get('factors', [])
    
    # Create dataframe from factors list
    df_factors = pd.json_normalize(factors_list)
    
    # Display settings and results
    print(f"\nFound {len(factors_list)} matching factors\n")
    pd.set_option('display.max_colwidth', None)
    display_summary(df_factors)
else:
    print(f"Error: {response.status_code}")
    print(f"Response: {response.text}")


Found 30 matching factors



,factorSet,source,activityType,activityUnit,publishedFrom,publishedTo,region,factorId,methodology,scope
0,NGERS,"National Greenhouse and Energy Reporting (Measurement) Determination 2008 (compiled 1 July 2023). Sourced from the Federal Register of Legislation at October 2023 and ongoing. For the latest information on Australian Government law please go to https://www.legislation.gov.au. Indirect Factor - NGA workbook (where applicable), published Aug 2023.",Liquefied Natural Gas (LNG),"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",2023-07-01,2024-06-30,Earth,158307,Activity-based,[1]
1,DEFRA,"2024 Greenhouse Gas Reporting: Conversion Factors 2024 (DEFRA) provided by gov.uk, license - https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",Liquefied Natural Gas (LNG),"[bbl, ccf, cf, gal, imp gal, kl, l, m3, mcf, ml]",2024-01-01,2024-12-31,Earth,172527,Activity-based,[1]
2,NGERS,"National Greenhouse and Energy Reporting (Measurement) Determination 2008 (compiled 1 July 2023). Sourced from the Federal Register of Legislation at October 2023 and ongoing. For the latest information on Australian Government law please go to https://www.legislation.gov.au. Indirect Factor - NGA workbook (where applicable), published Aug 2023.",Liquefied Natural Gas (LNG),"[bbl, ccf, cf, gal, imp gal, kl, l, m3, mcf, ml]",2023-07-01,2024-06-30,Earth,158682,Activity-based,[1]
3,DEFRA,"2024 Greenhouse Gas Reporting: Conversion Factors 2024 (DEFRA) provided by gov.uk, license - https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",Liquefied Natural Gas (LNG),"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",2024-01-01,2024-12-31,Earth,166157,Activity-based,[1]
4,DEFRA,"2024 Greenhouse Gas Reporting: Conversion Factors 2024 (DEFRA) provided by gov.uk, license - https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",Liquefied Natural Gas (LNG),"[bbl, ccf, cf, gal, imp gal, kl, l, m3, mcf, ml]",2024-01-01,2024-12-31,Earth,171164,Activity-based,[1]
5,DEFRA,"2024 Greenhouse Gas Reporting: Conversion Factors 2024 (DEFRA) provided by gov.uk, license - https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",Liquefied Natural Gas (LNG),"[g, kg, lb, million lb, st, t, thousand lb]",2024-01-01,2024-12-31,Earth,167943,Activity-based,[1]
6,EPA,Emission Factors for Greenhouse Gas Inventories Stationary Combustion 2024 by United States Environmental Protection Agency. Source: https://www.epa.gov/system/files/documents/2024-02/ghg-emission-factors-hub-2024.pdf,Liquefied Petroleum Gas (LPG),"[bbl, ccf, cf, gal, imp gal, kl, l, m3, mcf, ml]",2024-01-01,2024-12-31,Earth,500003326,Activity-based,[1]
7,EPA,Emission Factors for Greenhouse Gas Inventories Stationary Combustion 2024 by United States Environmental Protection Agency. Source: https://www.epa.gov/system/files/documents/2024-02/ghg-emission-factors-hub-2024.pdf,Liquefied Petroleum Gas (LPG),"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",2024-01-01,2024-12-31,Earth,500003263,Activity-based,[1]
8,US EPA/Climate Leaders + eGRID + USEEIO,US EPA 2024 EPA Centre for Climate Leadership. Emission Factors for Greenhouse Gas Inventories - https://www.epa.gov/climateleadership/ghg-emission-factors-hub,Liquefied Petroleum Gas (LPG),"[bbl, ccf, cf, gal, imp gal, kl, l, m3, mcf, ml]",2024-01-01,2024-12-31,Earth,162985,Activity-based,[1]
9,IPCC,"IPCC Guidelines for National Greenhouse Gas Inventories 2006, Volume 2, Stationary Combustion (corrected as of July 2023). Source: https://www.ipcc-nggip.iges.or.jp/public/2006gl/index.html",Derived Gases - Gas Works Gas,"[BTU, GJ, J, KJ, MJ, MMBTU, TJ, Wh, dth, kBTU, kWh, mWh, thm]",2006-07-01,NaN,Earth,500003499,Activity-based,[1]


### Output Description

The Search API returns a list of emission factors with the following fields:

<b>factorSet</b> - The name of the emission factor dataset or standard (e.g., "NGERS", "DEFRA", "EPA")

<b>source</b> - Detailed information about the data source, including the full name, version, publication details, and licensing information

<b>activityType</b> - The name/description of the activity or fuel type (e.g., "Liquefied Natural Gas (LNG)", "Natural Gas")

<b>activityUnit</b> - List of available units of measurement for the activity (e.g., ["BTU", "GJ", "kWh"] for energy, ["kg", "t"] for mass)

<b>publishedFrom</b> - The start date from which this emission factor is valid (format: YYYY-MM-DD)

<b>publishedTo</b> - The end date until which this emission factor is valid (format: YYYY-MM-DD). May be null for factors with no expiration

<b>region</b> - Geographic region where the factor applies (e.g., "Earth" for global factors, or specific country/region codes)

<b>factorId</b> - Unique identifier for the emission factor in the database

<b>methodology</b> - The calculation methodology used (e.g., "Activity-based", "Spend-based")

<b>scope</b> - The GHG Protocol scope(s) this factor applies to (e.g., [1] for Scope 1, [2] for Scope 2, [3.3] for Scope 3 category 3)

## Related Links

[Emissions API Developer Guide](https://developer.ibm.com/apis/catalog/ghgemissions--ibm-envizi-emissions-api/Introduction)